### Chapter 3: Coding Attention Mechanisms

#### 3.3 Attending to different parts of the input with self-attention

#### 3.3.1 A simple self-attention mechanism without trainable weights

In [1]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [2]:
input_query = inputs[1]
input_query

tensor([0.5500, 0.8700, 0.6600])

In [3]:
input_1 = inputs[0]
input_1

tensor([0.4300, 0.1500, 0.8900])

In [4]:
torch.dot(input_query, input_1)

tensor(0.9544)

In [6]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * input_query[idx]

print(res)

tensor(0.9544)


In [7]:
res = 0.
i = 3

res += torch.dot(input_query, input_1)

res

tensor(0.9544)

In [9]:
query = inputs[1] # 2nd input token is the query

attn_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, input_query)

attn_scores_2

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [12]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
attn_weights_2_tmp

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])

In [13]:
attn_weights_2_tmp.sum()

tensor(1.0000)

In [14]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

softmax_naive(attn_scores_2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [15]:
torch.softmax(attn_scores_2, dim=0)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])

In [16]:
context_vec_2 = torch.zeros(query.shape)

for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2_tmp[i] * x_i

context_vec_2

tensor([0.4355, 0.6451, 0.5680])

### 3.3.2 A simple self-attention mechanism without trainable weights

In [23]:
# attn_scores = torch.empty(len(inputs), len(inputs))

# for i, x_i in enumerate(inputs):
#     for j, x_j in enumerate(inputs):
#         attn_scores[i, j] = torch.dot(x_i, x_j)

# attn_scores

In [24]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=1)
attn_weights

# More efficient way to compute attention scores using matrix multiplication

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [28]:
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

### 3.4 Implementing self-attention with trainable weights

#### 3.4.1 Computing the attention weights step by step

In [50]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [51]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.randn(d_in, d_out))
W_key = torch.nn.Parameter(torch.randn(d_in, d_out))
W_value = torch.nn.Parameter(torch.randn(d_in, d_out))

In [52]:
query_2 = x_2 @ W_query
query_2

tensor([-1.1729, -0.0048], grad_fn=<SqueezeBackward4>)

In [54]:
keys = inputs @ W_key
values = inputs @ W_value

In [55]:
keys_2 = keys[1]

attn_score_22 = torch.dot(query_2, keys_2)
attn_score_22

tensor(0.1376, grad_fn=<DotBackward0>)

In [56]:
attn_score_2 = query_2 @ keys.T
attn_score_2


tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.3809],
       grad_fn=<SqueezeBackward4>)

In [58]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_score_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1704, 0.1611, 0.1652, 0.1412, 0.2505, 0.1117],
       grad_fn=<SoftmaxBackward0>)

In [59]:
attn_weights_2.sum()

tensor(1.0000, grad_fn=<SumBackward0>)

In [60]:
context_vec_2 = torch.sum(attn_weights_2.unsqueeze(-1) * values, dim=0)
context_vec_2

tensor([0.2854, 0.4081], grad_fn=<SumBackward1>)

#### 3.4.2 Implmenting a compact SelfAttention class

In [64]:
import torch.nn as nn

class SelfAttentionV1(nn.Module):
    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.W_query = nn.Parameter(torch.randn(d_in, d_out))
        self.W_key = nn.Parameter(torch.randn(d_in, d_out))
        self.W_value = nn.Parameter(torch.randn(d_in, d_out))

    def forward(self, x):
        queries = x @ self.W_query
        keys = x @ self.W_key
        values = x @ self.W_value
        
        d_k = keys.shape[-1]

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
        context = attn_weights @ values
        
        return context

torch.manual_seed(123)

sa_v1 = SelfAttentionV1(d_in, d_out)
sa_v1(inputs)

tensor([[0.2845, 0.4071],
        [0.2854, 0.4081],
        [0.2854, 0.4075],
        [0.2864, 0.3974],
        [0.2863, 0.3910],
        [0.2860, 0.4039]], grad_fn=<MmBackward0>)

In [66]:
import torch.nn as nn

class SelfAttentionV2(nn.Module):
    def __init__(self, d_in: int, d_out: int, qkv_bias: bool = False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)
        
        d_k = keys.shape[-1]

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
        context = attn_weights @ values
        
        return context

torch.manual_seed(1024)

sa_v2 = SelfAttentionV2(d_in, d_out)
sa_v2(inputs)

tensor([[-0.3460, -0.0092],
        [-0.3466, -0.0122],
        [-0.3466, -0.0121],
        [-0.3441, -0.0084],
        [-0.3441, -0.0095],
        [-0.3448, -0.0089]], grad_fn=<MmBackward0>)

### 3.5 Hiding future words with causal attention

#### 3.5.1 Applying a causal attention mask

In [67]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
values = sa_v2.W_value(inputs)

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)

attn_weights

tensor([[0.1474, 0.1648, 0.1645, 0.1785, 0.1642, 0.1806],
        [0.1361, 0.1622, 0.1619, 0.1863, 0.1656, 0.1879],
        [0.1362, 0.1622, 0.1619, 0.1863, 0.1657, 0.1878],
        [0.1497, 0.1644, 0.1643, 0.1772, 0.1665, 0.1779],
        [0.1455, 0.1635, 0.1633, 0.1800, 0.1673, 0.1804],
        [0.1479, 0.1643, 0.1641, 0.1783, 0.1658, 0.1794]],
       grad_fn=<SoftmaxBackward0>)

In [68]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

In [69]:
masked_simple = attn_weights * mask_simple
masked_simple

tensor([[0.1474, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1361, 0.1622, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1362, 0.1622, 0.1619, 0.0000, 0.0000, 0.0000],
        [0.1497, 0.1644, 0.1643, 0.1772, 0.0000, 0.0000],
        [0.1455, 0.1635, 0.1633, 0.1800, 0.1673, 0.0000],
        [0.1479, 0.1643, 0.1641, 0.1783, 0.1658, 0.1794]],
       grad_fn=<MulBackward0>)

In [70]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
normalized_masked_simple = masked_simple / row_sums
normalized_masked_simple

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4563, 0.5437, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2959, 0.3524, 0.3517, 0.0000, 0.0000, 0.0000],
        [0.2283, 0.2508, 0.2506, 0.2703, 0.0000, 0.0000],
        [0.1775, 0.1995, 0.1993, 0.2197, 0.2041, 0.0000],
        [0.1479, 0.1643, 0.1641, 0.1783, 0.1658, 0.1794]],
       grad_fn=<DivBackward0>)

In [75]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[-0.3107,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.5287, -0.2807,    -inf,    -inf,    -inf,    -inf],
        [-0.5279, -0.2806, -0.2832,    -inf,    -inf,    -inf],
        [-0.2849, -0.1522, -0.1535, -0.0464,    -inf,    -inf],
        [-0.3655, -0.2006, -0.2019, -0.0641, -0.1683,    -inf],
        [-0.3117, -0.1629, -0.1647, -0.0477, -0.1503, -0.0387]],
       grad_fn=<MaskedFillBackward0>)

In [76]:
attn_weights = torch.softmax(masked / d_k**0.5, dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4563, 0.5437, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2959, 0.3524, 0.3517, 0.0000, 0.0000, 0.0000],
        [0.2283, 0.2508, 0.2506, 0.2703, 0.0000, 0.0000],
        [0.1775, 0.1995, 0.1993, 0.2197, 0.2041, 0.0000],
        [0.1479, 0.1643, 0.1641, 0.1783, 0.1658, 0.1794]],
       grad_fn=<SoftmaxBackward0>)

#### 3.5.2 Masking additional attention weights with dropout

In [77]:
torch.manual_seed(123)

layer = nn.Dropout(0.5)
example = torch.ones(5, 5)
example

tensor([[1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.]])

In [78]:
layer(example)

tensor([[2., 2., 0., 2., 2.],
        [0., 0., 0., 0., 2.],
        [0., 2., 2., 2., 2.],
        [2., 0., 2., 0., 2.],
        [2., 0., 0., 2., 0.]])

#### 3.5.2 Implementing a compact causal self-attention class

In [79]:
batch = torch.stack((inputs, inputs), dim=0)
batch.shape

torch.Size([2, 6, 3])

In [85]:
class CausalAttention(nn.Module):
    def __init__(
        self,
        d_in: int,
        d_out: int,
        context_length: int,
        dropout: float = 0.5,
        qkv_bias: bool = False
    ):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.triu(
                torch.ones(context_length, context_length),
                diagonal=1
            )
        )

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        d_k = keys.shape[-1]

        attn_scores = queries @ keys.transpose(-2, -1)

        num_tokens = x.shape[1]

        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],
            -torch.inf
        )

        attn_weights = torch.softmax(
            attn_scores / d_k**0.5,
            dim=-1
        )

        attn_weights = self.dropout(attn_weights)

        context = attn_weights @ values

        return context
    
ca = CausalAttention(d_in, d_out, context_length=batch.shape[1], dropout=0.0)
ca(batch)

tensor([[[-0.2600, -0.1673],
         [-0.3039,  0.0894],
         [-0.3200,  0.1851],
         [-0.2851,  0.2075],
         [-0.2786,  0.2441],
         [-0.2656,  0.2377]],

        [[-0.2600, -0.1673],
         [-0.3039,  0.0894],
         [-0.3200,  0.1851],
         [-0.2851,  0.2075],
         [-0.2786,  0.2441],
         [-0.2656,  0.2377]]], grad_fn=<UnsafeViewBackward0>)

### 3.6 Extending single-head attention to multi-head attention

#### 3.6.1 Stacking multiple single-head attention layers

In [88]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in: int, d_out: int, context_length: int, num_heads: int = 2, dropout: float = 0.5, qkv_bias: bool = False):
        super().__init__()
        
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length=context_length, dropout=dropout, qkv_bias=qkv_bias) for _ in range(num_heads)]
        )
        
    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)

context_length = batch.shape[1]
d_in, d_out = 3, 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length=context_length, num_heads=2, qkv_bias=False)
mha(batch)

tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000],
         [-0.7381, -0.2026,  0.0000,  0.0000],
         [-0.9717, -0.2678,  0.9703,  0.7117],
         [-0.2210,  0.1084,  0.3492,  0.2569],
         [-0.4833, -0.1435,  0.4143,  0.3235],
         [-0.9130, -0.2881,  0.8947,  0.5938]],

        [[ 0.0000,  0.0000,  0.9544,  0.2127],
         [ 0.0000,  0.0000,  1.1781,  0.6513],
         [-0.7751,  0.0077,  0.4856,  0.3552],
         [-0.5504, -0.1770,  0.8795,  0.6697],
         [-0.8117, -0.1150,  0.4682,  0.3227],
         [-0.4257, -0.1539,  0.6311,  0.4446]]], grad_fn=<CatBackward0>)

#### 3.6.2 Implementing multi-head attention with weight splits

In [89]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<UnsafeViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
